In [ ]:
# Single agent -> one agent does everything
# one agent passes output to the next -> Sequential(pipeline)
# one main agent controls others -> Hierarchial(EG: Manager agent -> assigns tasks to workers) -> Manager-worker
# Agent communicate freely with each other -> Peer-to-peer
# Broadcast -> Request-Reply -> Pub/sub
# Orchestration in AI agents means managing and coordinating multiple components (agents, tools, models, workflows) so they
# work together smoothly to complete a task.

In [1]:
# Example 1: Sequential Multi-Agent System (Pipeline)
# Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”

# ================================
# AGENT 1: PLANNER
# ================================
def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]


# ================================
# AGENT 2: FLIGHT AGENT
# ================================
def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]


# ================================
# AGENT 3: WEATHER AGENT
# ================================
def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result


# RUN
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response)

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


In [2]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)

# ================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions

# ================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines

# ================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient
def intake_agent(patient_query):
    print("\n[Intake Agent] Collecting patient data...")
    
    patient_data = {
        "symptoms": ["fever", "cough"],
        "history": "No major illness"
    }
    
    plan = ["diagnostic", "treatment", "decision"]
    
    return patient_data, plan
def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Analyzing symptoms...")
    
    if "fever" in patient_data["symptoms"]:
        return {"condition": "Viral Infection", "severity": "Mild"}
    
    return {"condition": "Unknown", "severity": "Check further"}
def treatment_agent(diagnosis):
    print("\n[Treatment Agent] Suggesting treatment...")
    
    if diagnosis["condition"] == "Viral Infection":
        return ["Rest", "Hydration", "Paracetamol"]
    
    return ["Further Tests Required"]
def decision_agent(patient_data, diagnosis, treatment):
    print("\n[Decision Agent] Final recommendation...")
    
    return f"""
    Condition: {diagnosis['condition']}
    Severity: {diagnosis['severity']}
    Recommended Treatment: {', '.join(treatment)}
    """
def hospital_multi_agent(patient_query):
    print("Patient Query:", patient_query)
    
    patient_data, plan = intake_agent(patient_query)
    
    diagnosis = None
    treatment = None
    
    for step in plan:
        if step == "diagnostic":
            diagnosis = diagnostic_agent(patient_data)
        
        elif step == "treatment":
            treatment = treatment_agent(diagnosis)
        
        elif step == "decision":
            result = decision_agent(patient_data, diagnosis, treatment)
    
    return result
response = hospital_multi_agent("I have fever and cough")
print("\nFinal Recommendation:", response)

Patient Query: I have fever and cough

[Intake Agent] Collecting patient data...

[Diagnostic Agent] Analyzing symptoms...

[Treatment Agent] Suggesting treatment...

[Decision Agent] Final recommendation...

Final Recommendation: 
    Condition: Viral Infection
    Severity: Mild
    Recommended Treatment: Rest, Hydration, Paracetamol
    


In [8]:
import requests
import os
from dotenv import load_dotenv
# can usse groq for better results
load_dotenv()

API_TOKEN = os.getenv("my_key")

# print("API TOKEN:", API_TOKEN)

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

def query_llm(prompt: str):
    try:
        payload = {
            "model": MODEL_ID,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 150
        }

        response = requests.post(API_URL, headers=headers, json=payload, timeout=30)

        print("\n[STATUS]:", response.status_code)
        print("[RAW]:", response.text[:200])

        # Empty response
        if not response.text.strip():
            return "Error: Empty response"

        data = response.json()

        # Success
        if "choices" in data:
            return data["choices"][0]["message"]["content"]

        # Error case
        if "error" in data:
            return f"API Error: {data['error']}"

        return str(data)

    except Exception as e:
        return f"Error: {str(e)}"

def intake_agent(patient_query):
    print("\n[Intake Agent] Understanding patient...")

    thought = query_llm(f"Analyze patient symptoms: {patient_query}")
    print("LLM Thought:", thought)

    return {
        "symptoms": patient_query,
        "plan": ["diagnosis", "treatment", "decision"]
    }

def diagnostic_agent(symptoms):
    print("\n[Diagnostic Agent] Diagnosing...")

    diagnosis = query_llm(f"What could be the diagnosis for: {symptoms}?")
    print("LLM Diagnosis:", diagnosis)

    return {"diagnosis": diagnosis}

def treatment_agent(diagnosis):
    print("\n[Treatment Agent] Suggesting treatment...")

    treatment = query_llm(f"Suggest treatment for: {diagnosis}")
    print("LLM Treatment:", treatment)

    return {"treatment": treatment}

def decision_agent(diagnosis, treatment):
    print("\n[Decision Agent] Final decision...")

    final = query_llm(
        f"Based on diagnosis: {diagnosis} and treatment: {treatment}, give final recommendation."
    )

    return final

def hospital_multi_agent(patient_query):
    print("Patient Query:", patient_query)

    intake_data = intake_agent(patient_query)

    diagnosis = None
    treatment = None

    for step in intake_data["plan"]:

        if step == "diagnosis":
            diag_result = diagnostic_agent(intake_data["symptoms"])
            diagnosis = diag_result["diagnosis"]

        elif step == "treatment":
            treat_result = treatment_agent(diagnosis)
            treatment = treat_result["treatment"]

        elif step == "decision":
            result = decision_agent(diagnosis, treatment)

    return result
if __name__ == "__main__":
    response = hospital_multi_agent("Patient has fever and cough")
    print("\nFinal Answer:\n", response)

Patient Query: Patient has fever and cough

[Intake Agent] Understanding patient...

[STATUS]: 200
[RAW]: {"id":"b8ef581ce42644348004cbc2620c4a2d","object":"chat.completion","created":1773898035,"model":"meta-llama/llama-3-8b-instruct","choices":[{"index":0,"message":{"role":"assistant","content":"**Initi
LLM Thought: **Initial Assessment:**

Based on the patient's symptoms of fever and cough, several potential underlying causes can be considered. These symptoms are common in various respiratory infections, including:

1.  **Common Cold**: A viral infection caused by rhinoviruses, which often presents with a runny nose, sore throat, and cough.
2.  **Pneumonia**: A bacterial or viral infection that inflames the air sacs in the lungs, leading to fever, cough, and difficulty breathing.
3.  **Influenza (Flu)**: A highly contagious viral infection that causes fever, cough, sore throat, and body aches.
4.  **Bronchitis**: An inflammation of the bronchial tubes

[Diagnostic Agent] Diagnosing.

In [11]:
def manager_agent(goal):
    print("\n[Manager Agent] Planning tasks...")

    # Simple rule-based planning
    plan = []

    if "market" in goal.lower() or "product" in goal.lower():
        plan.append("market")

    if "budget" in goal.lower() or "feasibility" in goal.lower():
        plan.append("finance")

    plan.append("operations")
    plan.append("legal")
    plan.append("hr")

    print("Plan:", plan)
    return plan

def market_worker(goal):
    print("\n[Market Research Worker]")
    return "High demand in Tier-1 cities, moderate competition."


def finance_worker(goal):
    print("\n[Finance Worker]")
    return "Estimated $10M investment, ROI expected in 2 years."


def operations_worker(goal):
    print("\n[Operations Worker]")
    return "Factories can scale up, but shipping costs are high."


def legal_worker(goal):
    print("\n[Legal Worker]")
    return "Trademark available, but import laws require certification."


def hr_worker(goal):
    print("\n[HR Worker]")
    return "Need 50 new hires for customer support and sales."

def decision_agent(reports):
    print("\n[Decision Agent] Making final strategy...")

    decision = "Launch Product \n\n"

    # Add insights from reports
    decision += "Key Insights:\n"
    for key, value in reports.items():
        decision += f"- {key.upper()}: {value}\n"

    decision += "\nRisks:\n"
    decision += "- High shipping cost\n"
    decision += "- Legal certification delays\n"

    decision += "\nRecommendation:\n"
    decision += "Proceed with launch in Tier-1 cities with cost optimization."

    return decision

def corporate_multi_agent(goal):
    print("\nGoal:", goal)

    plan = manager_agent(goal)

    reports = {}

    for step in plan:

        if step == "market":
            reports["market"] = market_worker(goal)

        elif step == "finance":
            reports["finance"] = finance_worker(goal)

        elif step == "operations":
            reports["operations"] = operations_worker(goal)

        elif step == "legal":
            reports["legal"] = legal_worker(goal)

        elif step == "hr":
            reports["hr"] = hr_worker(goal)

    final_decision = decision_agent(reports)

    return final_decision
if __name__ == "__main__":
    result = corporate_multi_agent("Evaluate feasibility of launching Product Y in Asia")
    print("\nFINAL STRATEGY:\n")
    print(result)


Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Planning tasks...
Plan: ['market', 'finance', 'operations', 'legal', 'hr']

[Market Research Worker]

[Finance Worker]

[Operations Worker]

[Legal Worker]

[HR Worker]

[Decision Agent] Making final strategy...

FINAL STRATEGY:

Launch Product 

Key Insights:
- MARKET: High demand in Tier-1 cities, moderate competition.
- FINANCE: Estimated $10M investment, ROI expected in 2 years.
- OPERATIONS: Factories can scale up, but shipping costs are high.
- LEGAL: Trademark available, but import laws require certification.
- HR: Need 50 new hires for customer support and sales.

Risks:
- High shipping cost
- Legal certification delays

Recommendation:
Proceed with launch in Tier-1 cities with cost optimization.


In [13]:
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market. The Manager Agent oversees the process and delegates tasks to specialized workers.

# 👩‍💼 Manager Agent
# - Receives the overall goal: “Evaluate feasibility of launching Product Y in Asia.”
# - Dynamically assigns tasks to worker agents depending on what’s needed.
# - Example: If budget is unclear → send to Finance Worker. If regulations are complex → send to Legal Worker.

# ================================
# 📊 Worker Agents
# ================================
# - Market Research Worker
# - Collects competitor data, customer preferences, and demand forecasts.
# - Reports: “High demand in Tier‑1 cities, moderate competition.”
# - Finance Worker
# - Analyzes budget, ROI, and pricing strategy.
# - Reports: “Estimated $10M investment, ROI in 2 years.”
# - Operations Worker
# - Evaluates supply chain, production capacity, and logistics.
# - Reports: “Factories can scale up, but shipping costs are high.”
# - Legal Worker
# - Reviews compliance, intellectual property, and regional regulations.
# - Reports: “Trademark available, but import laws require certification.”
# - HR Worker
# - Assesses staffing needs and training requirements.
# - Reports: “Need 50 new hires for customer support and sales.”
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_TOKEN = os.getenv("my_key")
API_URL = "https://router.huggingface.co/v1/chat/completions"
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}
def query_llm(prompt):
    try:
        payload = {
            "model": MODEL_ID,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 200
        }

        response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=30)

        print("\n[STATUS]:", response.status_code)

        if not response.text.strip():
            return "Error: Empty response"

        data = response.json()

        if "choices" in data:
            return data["choices"][0]["message"]["content"]

        if "error" in data:
            return f"API Error: {data['error']}"

        return str(data)

    except Exception as e:
        return f"Error: {str(e)}"

def manager_agent(goal):
    print("\n[Manager Agent] Planning tasks...")

    prompt = f"""
    Goal: {goal}

    Decide which workers are needed from:
    Market Research, Finance, Operations, Legal, HR.

    Return list like:
    ["market", "finance", "operations", "legal", "hr"]
    """

    plan_text = query_llm(prompt)
    print("Plan:", plan_text)

    return ["market", "finance", "operations", "legal", "hr"]

def market_worker(goal):
    print("\n[Market Research Worker]")
    return query_llm(f"Analyze market demand, competitors for: {goal}")


def finance_worker(goal):
    print("\n[Finance Worker]")
    return query_llm(f"Estimate budget, ROI for: {goal}")


def operations_worker(goal):
    print("\n[Operations Worker]")
    return query_llm(f"Analyze supply chain and logistics for: {goal}")


def legal_worker(goal):
    print("\n[Legal Worker]")
    return query_llm(f"Check regulations, compliance for: {goal}")


def hr_worker(goal):
    print("\n[HR Worker]")
    return query_llm(f"Estimate hiring and staffing needs for: {goal}")
def decision_agent(reports):
    print("\n[Decision Agent] Final Strategy...")

    prompt = f"""
    Based on these reports:

    {reports}

    Give final business decision:
    - Launch or not
    - Risks
    - Recommendations
    """

    return query_llm(prompt)
def corporate_multi_agent(goal):
    print("\nGoal:", goal)

    plan = manager_agent(goal)

    reports = {}

    for step in plan:

        if step == "market":
            reports["market"] = market_worker(goal)

        elif step == "finance":
            reports["finance"] = finance_worker(goal)

        elif step == "operations":
            reports["operations"] = operations_worker(goal)

        elif step == "legal":
            reports["legal"] = legal_worker(goal)

        elif step == "hr":
            reports["hr"] = hr_worker(goal)

    final_decision = decision_agent(reports)

    return final_decision
if __name__ == "__main__":
    result = corporate_multi_agent("Evaluate feasibility of launching Product Y in Asia")
    print("\n FINAL STRATEGY:\n", result)


Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Planning tasks...

[STATUS]: 200
Plan: To evaluate the feasibility of launching Product Y in Asia, we would need a team with diverse skill sets. Here's a suggested list of workers needed:

- **Marketing Research**: To understand the Asian market, consumer behavior, and competitor analysis.
- **Finance**: To analyze the financial aspects of launching a product in a new market, including budgeting, funding, and potential return on investment.
- **Operations**: To manage the logistics of launching the product, including supply chain, distribution, and inventory management.
- **Legal**: To ensure compliance with local laws and regulations, including intellectual property protection, labor laws, and tax requirements.
- **HR**: To manage the local workforce, including recruitment, training, and employee relations.

Here's the list of workers needed:
```python
required_workers = ["market", "finance", "operations", "le

In [34]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("API Key not found!")

print("Key Loaded:", API_KEY[:10], "...")

client = Groq(api_key=API_KEY)
MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.1-70b-versatile",
    "llama-3.2-90b-vision-preview"
]


def query_llm(prompt):
    for model in MODELS:
        try:
            print(f"Trying model: {model}")

            res = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5
            )

            return res.choices[0].message.content.strip()

        except Exception as e:
            print(f"{model} failed")

    return "All models failed"

def manager_agent(goal):
    print("\n[Manager Agent] Planning...")

    prompt = f"""
    Goal: {goal}

    Decide required departments:
    market, finance, operations, legal, hr

    Return ONLY comma-separated list.
    """

    plan_text = query_llm(prompt)
    print("Plan:", plan_text)

    return plan_text.lower().replace(" ", "").split(",")

def market_worker(goal):
    print("\n[Market Worker]")
    return query_llm(f"Analyze demand & competition: {goal}")

def finance_worker(goal):
    print("\n[Finance Worker]")
    return query_llm(f"Estimate budget & ROI: {goal}")

def operations_worker(goal):
    print("\n[Operations Worker]")
    return query_llm(f"Analyze supply chain & logistics: {goal}")

def legal_worker(goal):
    print("\n[Legal Worker]")
    return query_llm(f"Check legal compliance: {goal}")

def hr_worker(goal):
    print("\n[HR Worker]")
    return query_llm(f"Estimate hiring needs: {goal}")

def decision_agent(results):
    print("\n[Decision Agent] Final Strategy...")

    prompt = f"""
    Market: {results.get('market')}
    Finance: {results.get('finance')}
    Operations: {results.get('operations')}
    Legal: {results.get('legal')}
    HR: {results.get('hr')}

    Final decision: Should company launch or not? Give reason.
    """

    return query_llm(prompt)

def run_system(goal):
    print("Goal:", goal)

    plan = manager_agent(goal)
    results = {}

    for step in plan:
        if "market" in step:
            results["market"] = market_worker(goal)

        elif "finance" in step:
            results["finance"] = finance_worker(goal)

        elif "operations" in step:
            results["operations"] = operations_worker(goal)

        elif "legal" in step:
            results["legal"] = legal_worker(goal)

        elif "hr" in step:
            results["hr"] = hr_worker(goal)

    final = decision_agent(results)
    return final

goal = "Evaluate feasibility of launching Product Y in Asia"

output = run_system(goal)

print("\n================ FINAL OUTPUT ================\n")
print(output)

Key Loaded: gsk_qa2Sah ...
Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Planning...
Trying model: llama-3.1-8b-instant
Plan: market, finance, operations, legal, HR

[Market Worker]
Trying model: llama-3.1-8b-instant

[Finance Worker]
Trying model: llama-3.1-8b-instant

[Operations Worker]
Trying model: llama-3.1-8b-instant

[Legal Worker]
Trying model: llama-3.1-8b-instant

[HR Worker]
Trying model: llama-3.1-8b-instant

[Decision Agent] Final Strategy...
Trying model: llama-3.1-8b-instant

================ FINAL OUTPUT ================

Based on the comprehensive analysis of the market, competition, regulatory compliance, intellectual property protection, hiring needs, and feasibility evaluation, I recommend that the company launch Product Y in Asia.

Here's a summary of the key reasons for launching Product Y in Asia:

**Market demand**: The market research indicates a significant demand for Product Y in Asia, with a potential market size of $100 million

In [35]:
# Agent 1: Crisis Coordinator (Broadcaster)
# - Broadcasts: “Data breach detected in customer database. Immediate response required.”
# - Sends this to all other agents simultaneously.

# 🛡️ Agent 2: IT Security Agent
# - Receives broadcast.
# - Responds: “Isolate affected servers, patch vulnerabilities, start forensic analysis.”

# 📞 Agent 3: Communications Agent
# - Receives broadcast.
# - Responds: “Draft internal memo, prepare press release, notify stakeholders.”

# 💰 Agent 4: Finance Agent
# - Receives broadcast.
# - Responds: “Estimate financial impact, allocate emergency funds, review insurance coverage.”

# 👩‍⚖️ Agent 5: Legal Agent
# - Receives broadcast.
# - Responds: “Assess regulatory obligations, prepare compliance reports, advise on liability.”

# 👩‍💼 Agent 6: HR Agent
# - Receives broadcast.
# - Responds: “Brief employees, provide guidance on handling customer queries, ensure morale support.”

# 🧑‍⚖️ Agent 7: Decision Agent (Coordinator)
# - Collects all responses.
# - Integrates into a final crisis response plan:
# “Servers isolated, communications prepared, financial impact assessed, compliance secured, employees briefed.”
def crisis_coordinator():
    print("\n[Crisis Coordinator]")
    message = "Data breach detected in customer database. Immediate response required."
    print("Broadcasting Message:", message)
    return message

def it_security_agent(message):
    print("\n[IT Security Agent]")
    return "Isolate affected servers, patch vulnerabilities, start forensic analysis."

def communications_agent(message):
    print("\n[Communications Agent]")
    return "Draft internal memo, prepare press release, notify stakeholders."

def finance_agent(message):
    print("\n[Finance Agent]")
    return "Estimate financial impact, allocate emergency funds, review insurance coverage."

def legal_agent(message):
    print("\n[Legal Agent]")
    return "Assess regulatory obligations, prepare compliance reports, advise on liability."

def hr_agent(message):
    print("\n[HR Agent]")
    return "Brief employees, guide customer handling, ensure morale support."

def decision_agent(responses):
    print("\n[Decision Agent] Creating final plan...\n")

    final_plan = f"""
    FINAL CRISIS RESPONSE PLAN 

    IT: {responses['it']}
    Communications: {responses['comm']}
    Finance: {responses['finance']}
    Legal: {responses['legal']}
    HR: {responses['hr']}

    Summary:
    Servers isolated, communications prepared, financial impact assessed,
    compliance secured, employees briefed.
    """

    return final_plan

def crisis_management_system():
    message = crisis_coordinator()
    responses = {}

    responses["it"] = it_security_agent(message)
    responses["comm"] = communications_agent(message)
    responses["finance"] = finance_agent(message)
    responses["legal"] = legal_agent(message)
    responses["hr"] = hr_agent(message)

    final_output = decision_agent(responses)

    return final_output

output = crisis_management_system()
print(output)


[Crisis Coordinator]
Broadcasting Message: Data breach detected in customer database. Immediate response required.

[IT Security Agent]

[Communications Agent]

[Finance Agent]

[Legal Agent]

[HR Agent]

[Decision Agent] Creating final plan...


    FINAL CRISIS RESPONSE PLAN 

    IT: Isolate affected servers, patch vulnerabilities, start forensic analysis.
    Communications: Draft internal memo, prepare press release, notify stakeholders.
    Finance: Estimate financial impact, allocate emergency funds, review insurance coverage.
    Legal: Assess regulatory obligations, prepare compliance reports, advise on liability.
    HR: Brief employees, guide customer handling, ensure morale support.

    Summary:
    Servers isolated, communications prepared, financial impact assessed,
    compliance secured, employees briefed.
    


In [38]:
# Scenario: Corporate Product Launch Broadcast
# Imagine a company preparing to launch Product X in Q3. The Coordinator Agent (like a corporate program manager) sends out a broadcast announcement to all departments at once:
# “Product X launch in Q3, target market North America, budget $5M.”


# 📢 Coordinator Agent (Broadcaster)
# - Sends the launch announcement to all departments simultaneously.
# - This is implemented in the code by the broadcast() function, which uses asyncio.gather() to run all agents in parallel.

# 📈 Marketing Agent
# - Receives the broadcast.
# - Responds with a marketing strategy: campaigns, channels, and positioning.
# 💰 Finance Agent
# - Receives the broadcast.
# - Responds with budget allocation and ROI forecasts.
# 🏭 Operations Agent
# - Receives the broadcast.
# - Responds with production and supply chain actions.
# 👩‍⚖️ Legal Agent
# - Receives the broadcast.
# - Responds with compliance checks and contract actions.
# 👩‍💼 HR Agent
# - Receives the broadcast.
# - Responds with staffing and training plans.

# 🧑‍⚖️ Decision Agent
# - Collects all responses.
# - Integrates them into a Final Corporate Launch Plan.
# - In the code, this is the decision_agent() function that merges all outputs into one consolidated plan.
import asyncio
import nest_asyncio

nest_asyncio.apply()

async def coordinator():
    message = "Product X launch in Q3, target market North America, budget 5M"
    print("\nCoordinator Agent")
    print("Broadcast:", message)
    return message

async def marketing_agent(msg):
    print("\nMarketing Agent")
    return "Campaigns on social media, influencer marketing, premium positioning"

async def finance_agent(msg):
    print("\nFinance Agent")
    return "Allocate 5M budget, expected ROI in 18 months"

async def operations_agent(msg):
    print("\nOperations Agent")
    return "Increase production capacity, optimize supply chain for North America"

async def legal_agent(msg):
    print("\nLegal Agent")
    return "Ensure compliance with regional laws, finalize contracts"

async def hr_agent(msg):
    print("\nHR Agent")
    return "Hire marketing and sales staff, conduct training programs"

async def decision_agent(results):
    print("\nDecision Agent")
    final = f"""
Marketing: {results[0]}
Finance: {results[1]}
Operations: {results[2]}
Legal: {results[3]}
HR: {results[4]}

Final Plan:
Launch Product X in Q3 with aligned strategy, budget, operations, compliance, and workforce
"""
    return final

async def broadcast_system():
    msg = await coordinator()

    responses = await asyncio.gather(
        marketing_agent(msg),
        finance_agent(msg),
        operations_agent(msg),
        legal_agent(msg),
        hr_agent(msg)
    )

    final = await decision_agent(responses)
    return final


output = await broadcast_system()

print("\nFinal Output: ")
print(output)


Coordinator Agent
Broadcast: Product X launch in Q3, target market North America, budget 5M

Marketing Agent

Finance Agent

Operations Agent

Legal Agent

HR Agent

Decision Agent

Final Output: 

Marketing: Campaigns on social media, influencer marketing, premium positioning
Finance: Allocate 5M budget, expected ROI in 18 months
Operations: Increase production capacity, optimize supply chain for North America
Legal: Ensure compliance with regional laws, finalize contracts
HR: Hire marketing and sales staff, conduct training programs

Final Plan:
Launch Product X in Q3 with aligned strategy, budget, operations, compliance, and workforce

